In [11]:
%pip install pandas numpy scikit-learn nltk ftfy shap nbformat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [nbformat]
Note: you may need to restart the kernel to use updated packages.


In [2]:
%run 'MMDA_linear_ensemble.ipynb'
clear_output()

In [21]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import scipy.sparse as sparse
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
MODEL_PATH = "saved_final_models/final_models_20260615_104839"

if not os.path.exists(MODEL_PATH):
    print(f"ERROR: Path {MODEL_PATH} does not exist!")
    print("Available directories in saved_final_models:")
    if os.path.exists("saved_final_models"):
        for d in os.listdir("saved_final_models"):
            print(f"  - saved_final_models/{d}")
    else:
        print("  No saved_final_models directory found!")
    raise FileNotFoundError(f"Please update MODEL_PATH to the correct directory")

SHAP_BASE_DIR = f"{MODEL_PATH}/shap_analysis_results"
os.makedirs(SHAP_BASE_DIR, exist_ok=True)

print("Loading preprocessing objects...")
preprocessing = joblib.load(f"{MODEL_PATH}/preprocessing_objects.joblib")
feature_names = joblib.load(f"{MODEL_PATH}/feature_names.joblib")

print("Loading data...")
X_train = sparse.load_npz(f"{MODEL_PATH}/X_train.npz")
X_test = sparse.load_npz(f"{MODEL_PATH}/X_test.npz")
X_train_dense = np.load(f"{MODEL_PATH}/X_train_dense.npy")
X_test_dense = np.load(f"{MODEL_PATH}/X_test_dense.npy")
y_test = np.load(f"{MODEL_PATH}/y_test.npy")

label_encoder = preprocessing['label_encoder']
class_names = label_encoder.classes_

print(f"X_test shape: {X_test.shape}")
print(f"X_test_dense shape: {X_test_dense.shape}")
print(f"Number of classes: {len(class_names)}")
print(f"Classes: {class_names}")

Loading preprocessing objects...
Loading data...
X_test shape: (4224, 12774)
X_test_dense shape: (4224, 310)
Number of classes: 6
Classes: ['age' 'ethnicity' 'gender' 'not_cyberbullying' 'other_cyberbullying'
 'religion']


In [25]:
models = {}

print("Loading models...")

models['Logistic Regression'] = joblib.load(f"{MODEL_PATH}/Logistic_Regression.joblib")
models['Linear SVM'] = joblib.load(f"{MODEL_PATH}/Linear_SVM.joblib")
models['SGD Linear SVM'] = joblib.load(f"{MODEL_PATH}/SGD_Linear_SVM.joblib")
models['Random Forest'] = joblib.load(f"{MODEL_PATH}/Random_Forest_on_SVD_features.joblib")
models['Extra Trees'] = joblib.load(f"{MODEL_PATH}/Extra_Trees_on_SVD_features.joblib")
models['HistGradientBoosting'] = joblib.load(f"{MODEL_PATH}/HistGradientBoosting_on_SVD_features.joblib")
models['Voting Ensemble'] = joblib.load(f"{MODEL_PATH}/Voting_Ensemble.joblib")

print(f"Total models loaded: {len(models)}")

Loading models...
Total models loaded: 7


In [29]:
def plot_coefficient_importance(model, model_name, X_sample_dense, feature_names_list, model_folder):
    print(f"Analyzing coefficients for {model_name}...")
    
    if hasattr(model, 'coef_'):
        coef = model.coef_
        if len(coef.shape) == 2:
            importances = np.mean(np.abs(coef), axis=0)
        else:
            importances = np.abs(coef)
        
        if len(importances) > X_sample_dense.shape[1]:
            importances = importances[:X_sample_dense.shape[1]]
        
        top_n = 30
        top_indices = np.argsort(importances)[-top_n:]
        
        plt.figure(figsize=(12, 8))
        plt.barh(range(top_n), importances[top_indices])
        plt.yticks(range(top_n), [feature_names_list[i][:40] for i in top_indices])
        plt.xlabel("Absolute Coefficient Value")
        plt.title(f"{model_name} - Top {top_n} Features by Coefficient Magnitude")
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, "coefficient_importance.png"), dpi=150, bbox_inches='tight')
        plt.close()
        
        importance_df = pd.DataFrame({
            'feature': [feature_names_list[i] for i in top_indices],
            'coefficient': importances[top_indices]
        }).sort_values('coefficient', ascending=False)
        importance_df.to_csv(os.path.join(model_folder, "coefficient_importance.csv"), index=False)
        
        print(f"Coefficient importance saved to {model_folder}")
        return True
    else:
        print(f"No coef_ attribute found for {model_name}")
        return False

## Linear models

In [ ]:
def analyze_linear_model(model, model_name, X_sample, feature_names_list):
    print(f"Analyzing: {model_name}")
    
    model_folder = os.path.join(SHAP_BASE_DIR, model_name.replace(' ', '_'))
    os.makedirs(model_folder, exist_ok=True)
    
    if hasattr(X_sample, 'toarray'):
        X_sample_dense = X_sample.toarray()
    else:
        X_sample_dense = X_sample

    if not hasattr(model, 'predict_proba'):
        print(f"Model doesn't have predict_proba, using coefficient analysis")
        return plot_coefficient_importance(model, model_name, X_sample_dense, feature_names_list, model_folder)

    print("Creating LinearExplainer...")
    explainer = shap.LinearExplainer(model, X_sample_dense[:100], feature_perturbation="interventional")
    
    print("Computing SHAP values...")
    shap_values = explainer.shap_values(X_sample_dense[:50])
    
    joblib.dump(shap_values, os.path.join(model_folder, "shap_values.joblib"))
    print("SHAP values saved")
    
    if len(shap_values.shape) == 3:
        for i, class_name in enumerate(class_names):
            if i < shap_values.shape[2]:
                plt.figure(figsize=(10, 6))
                shap.summary_plot(shap_values[:, :, i], X_sample_dense[:50],
                                feature_names=feature_names_list[:X_sample_dense.shape[1]],
                                show=False, max_display=20)
                plt.title(f"{model_name} - Class: {class_name}")
                plt.tight_layout()
                plt.savefig(os.path.join(model_folder, f"shap_summary_{class_name}.png"), 
                            dpi=150, bbox_inches='tight')
                plt.close()
    elif len(shap_values.shape) == 2:
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_sample_dense[:50],
                        feature_names=feature_names_list[:X_sample_dense.shape[1]],
                        show=False, max_display=20)
        plt.title(f"{model_name} - SHAP Summary")
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, "shap_summary.png"), dpi=150, bbox_inches='tight')
        plt.close()
    
    print(f"Analysis complete for {model_name}")
    return True

for model_name in ['Logistic Regression', 'Linear SVM']:
    X_sample = X_test[:100]
    feature_names_list = feature_names['all_features']
    analyze_linear_model(models[model_name], model_name, X_sample, feature_names_list)

Analyzing: Logistic Regression
Creating LinearExplainer...
Computing SHAP values...
SHAP values saved
Analysis complete for Logistic Regression
Analyzing: Linear SVM
Model doesn't have predict_proba, using coefficient analysis
Analyzing coefficients for Linear SVM...
Coefficient importance saved to saved_final_models/final_models_20260615_104839/shap_analysis_results/Linear_SVM


In [ ]:
def analyze_sgd_model(model, model_name, X_sample, feature_names_list):
    print(f"Analyzing: {model_name}")
    
    model_folder = os.path.join(SHAP_BASE_DIR, model_name.replace(' ', '_'))
    os.makedirs(model_folder, exist_ok=True)
    
    if hasattr(X_sample, 'toarray'):
        X_sample_dense = X_sample.toarray()
    else:
        X_sample_dense = X_sample
    
    # For SGD, use coefficient importance since predict_proba is not available
    print("Using coefficient importance analysis...")
    success = plot_coefficient_importance(model, model_name, X_sample_dense, feature_names_list, model_folder)
    print(f"Analysis complete for {model_name}")
    
    return success

X_sample = X_test[:100]
feature_names_list = feature_names['all_features']
analyze_sgd_model(models['SGD Linear SVM'], 'SGD Linear SVM', X_sample, feature_names_list)

Analyzing: SGD Linear SVM
Using coefficient importance analysis...
Analyzing coefficients for SGD Linear SVM...
Coefficient importance saved to saved_final_models/final_models_20260615_104839/shap_analysis_results/SGD_Linear_SVM
Analysis complete for SGD Linear SVM


## Tree based

In [45]:
def analyze_tree_model(model, model_name, X_sample, feature_names_list):
    print(f"Analyzing: {model_name}")
    
    model_folder = os.path.join(SHAP_BASE_DIR, model_name.replace(' ', '_'))
    os.makedirs(model_folder, exist_ok=True)
    
    try:
        print("Attempting TreeExplainer...")
        explainer = shap.TreeExplainer(model)
        
        print("Computing SHAP values...")
        n_samples = min(30, X_sample.shape[0])
        shap_values = explainer.shap_values(X_sample[:n_samples])
        
        joblib.dump(shap_values, os.path.join(model_folder, "shap_values.joblib"))

        print("Creating summary plots...")
        if isinstance(shap_values, list):
            for i, class_name in enumerate(class_names):
                if i < len(shap_values):
                    plt.figure(figsize=(10, 6))
                    n_features_to_show = min(20, X_sample.shape[1])
                    shap.summary_plot(shap_values[i][:, :n_features_to_show], 
                                    X_sample[:n_samples, :n_features_to_show],
                                    feature_names=feature_names_list[:n_features_to_show],
                                    show=False, max_display=20)
                    plt.title(f"{model_name} - Class: {class_name}")
                    plt.tight_layout()
                    plt.savefig(os.path.join(model_folder, f"shap_summary_{class_name}.png"), 
                              dpi=150, bbox_inches='tight')
                    plt.close()
        else:
            plt.figure(figsize=(10, 6))
            n_features_to_show = min(20, X_sample.shape[1])
            shap.summary_plot(shap_values[:n_samples, :n_features_to_show], 
                            X_sample[:n_samples, :n_features_to_show],
                            feature_names=feature_names_list[:n_features_to_show],
                            show=False, max_display=20)
            plt.title(f"{model_name} - SHAP Summary")
            plt.tight_layout()
            plt.savefig(os.path.join(model_folder, "shap_summary.png"), dpi=150, bbox_inches='tight')
            plt.close()
        
        if hasattr(model, 'feature_importances_'):
            print("  Computing feature importance...")
            importances = model.feature_importances_
            if len(importances) > X_sample.shape[1]:
                importances = importances[:X_sample.shape[1]]
            
            top_n = min(30, len(importances))
            top_indices = np.argsort(importances)[-top_n:]
            
            plt.figure(figsize=(12, 8))
            plt.barh(range(top_n), importances[top_indices])
            plt.yticks(range(top_n), [feature_names_list[i][:40] for i in top_indices])
            plt.xlabel("Feature Importance")
            plt.title(f"{model_name} - Top {top_n} Features")
            plt.tight_layout()
            plt.savefig(os.path.join(model_folder, "feature_importance.png"), dpi=150, bbox_inches='tight')
            plt.close()
            
            importance_df = pd.DataFrame({
                'feature': [feature_names_list[i] for i in top_indices],
                'importance': importances[top_indices]
            }).sort_values('importance', ascending=False)
            importance_df.to_csv(os.path.join(model_folder, "feature_importance.csv"), index=False)
            print("Feature importance saved")
        
    except Exception as e:
        print(f"TreeExplainer failed: {e}")
        print("Falling back to KernelExplainer...")
        return analyze_tree_with_kernel(model, model_name, X_sample, feature_names_list)

def analyze_tree_with_kernel(model, model_name, X_sample, feature_names_list):
    print(f"Using KernelExplainer...")
    
    model_folder = os.path.join(SHAP_BASE_DIR, model_name.replace(' ', '_'))
    os.makedirs(model_folder, exist_ok=True)
    
    n_background = min(20, X_sample.shape[0])
    background = X_sample[:n_background]
    
    if hasattr(model, 'predict_proba'):
        predict_fn = model.predict_proba
    else:
        predict_fn = model.predict
    
    print(f"Creating KernelExplainer with {n_background} background samples...")
    explainer = shap.KernelExplainer(predict_fn, background)
    
    n_samples = min(10, X_sample.shape[0])
    print(f"Computing SHAP values for {n_samples} samples...")
    shap_values = explainer.shap_values(X_sample[:n_samples])
    
    joblib.dump(shap_values, os.path.join(model_folder, "kernel_shap_values.joblib"))
    
    print("Creating summary plots...")
    if isinstance(shap_values, list):
        for i, class_name in enumerate(class_names):
            if i < len(shap_values):
                plt.figure(figsize=(10, 6))
                n_features_to_show = min(20, X_sample.shape[1])
                shap.summary_plot(shap_values[i][:, :n_features_to_show], 
                                X_sample[:n_samples, :n_features_to_show],
                                feature_names=feature_names_list[:n_features_to_show],
                                show=False, max_display=20)
                plt.title(f"{model_name} (Kernel) - Class: {class_name}")
                plt.tight_layout()
                plt.savefig(os.path.join(model_folder, f"kernel_shap_summary_{class_name}.png"), 
                          dpi=150, bbox_inches='tight')
                plt.close()
    else:
        plt.figure(figsize=(10, 6))
        n_features_to_show = min(20, X_sample.shape[1])
        shap.summary_plot(shap_values[:, :n_features_to_show], 
                        X_sample[:n_samples, :n_features_to_show],
                        feature_names=feature_names_list[:n_features_to_show],
                        show=False, max_display=20)
        plt.title(f"{model_name} (Kernel) - SHAP Summary")
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, "kernel_shap_summary.png"), dpi=150, bbox_inches='tight')
        plt.close()
    
    if hasattr(model, 'feature_importances_'):
        print("Computing model feature importance...")
        importances = model.feature_importances_
        if len(importances) > X_sample.shape[1]:
            importances = importances[:X_sample.shape[1]]
        
        top_n = min(30, len(importances))
        top_indices = np.argsort(importances)[-top_n:]
        
        plt.figure(figsize=(12, 8))
        plt.barh(range(top_n), importances[top_indices])
        plt.yticks(range(top_n), [feature_names_list[i][:40] for i in top_indices])
        plt.xlabel("Feature Importance")
        plt.title(f"{model_name} - Top {top_n} Features")
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, "feature_importance.png"), dpi=150, bbox_inches='tight')
        plt.close()
        
        importance_df = pd.DataFrame({
            'feature': [feature_names_list[i] for i in top_indices],
            'importance': importances[top_indices]
        }).sort_values('importance', ascending=False)
        importance_df.to_csv(os.path.join(model_folder, "feature_importance.csv"), index=False)
        print("Feature importance saved")
    
    print(f"KernelExplainer analysis complete for {model_name}")
    return True

for model_name in ['Random Forest', 'Extra Trees', 'HistGradientBoosting']:
    X_sample = X_test_dense[:100]
    n_svd_features = X_test_dense.shape[1] - 10
    feature_names_list = [f"svd_{i}" for i in range(n_svd_features)] + feature_names['meta_features'][:10]
    analyze_tree_model(models[model_name], model_name, X_sample, feature_names_list)

Analyzing: Random Forest
Attempting TreeExplainer...
Computing SHAP values...
Creating summary plots...
  Computing feature importance...
Feature importance saved
Analyzing: Extra Trees
Attempting TreeExplainer...
Computing SHAP values...
Creating summary plots...
  Computing feature importance...
Feature importance saved
Analyzing: HistGradientBoosting
Attempting TreeExplainer...
Computing SHAP values...
Creating summary plots...


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

## Ensemble model

In [42]:
def analyze_voting_ensemble(model, model_name):
    print(f"Analyzing: {model_name}")
    
    model_folder = os.path.join(SHAP_BASE_DIR, model_name.replace(' ', '_'))
    os.makedirs(model_folder, exist_ok=True)
    
    estimators = model.estimators_
    estimator_names = ['LogisticRegression', 'LinearSVC', 'SGDClassifier']
    
    success_count = 0
    
    for est_name, estimator in zip(estimator_names, estimators):
        print(f"Analyzing estimator: {est_name}")
        
        est_folder = os.path.join(model_folder, est_name)
        os.makedirs(est_folder, exist_ok=True)
        
        X_sample = X_test[:100]
        feature_names_list = feature_names['all_features']
        
        if hasattr(X_sample, 'toarray'):
            X_sample_dense = X_sample.toarray()
        else:
            X_sample_dense = X_sample
        
        if est_name == 'LogisticRegression':
            print(f"Analyzing {est_name} with multiple methods...")
            
            explainer = shap.LinearExplainer(estimator, X_sample_dense[:100])
            shap_values = explainer.shap_values(X_sample_dense[:50])
            
            # Save results
            joblib.dump(shap_values, os.path.join(est_folder, "shap_values.joblib"))
            
            for i, class_name in enumerate(class_names):
                if i < shap_values.shape[2]:
                    plt.figure(figsize=(10, 6))
                    shap.summary_plot(shap_values[:, :, i], X_sample_dense[:50],
                                    feature_names=feature_names_list[:X_sample_dense.shape[1]],
                                    show=False, max_display=20)
                    plt.title(f"{model_name} - {est_name} - Class: {class_name}")
                    plt.tight_layout()
                    plt.savefig(os.path.join(est_folder, f"shap_summary_{class_name}.png"), 
                                dpi=150, bbox_inches='tight')
                    plt.close()

            success_count += 1

        elif est_name == 'LinearSVC':
            print(f"Analyzing {est_name} with coefficient and permutation importance...")
            
            if plot_coefficient_importance(estimator, f"{model_name}_{est_name}", 
                                         X_sample_dense, feature_names_list, est_folder):
                success_count += 1
        
        elif est_name == 'SGDClassifier':
            print(f"Analyzing {est_name} with coefficient importance...")
            if plot_coefficient_importance(estimator, f"{model_name}_{est_name}", 
                                         X_sample_dense, feature_names_list, est_folder):
                success_count += 1
    
    print(f"Analyzed {success_count}/{len(estimators)} estimators successfully")

analyze_voting_ensemble(models['Voting Ensemble'], 'Voting Ensemble')

Analyzing: Voting Ensemble
Analyzing estimator: LogisticRegression
Analyzing LogisticRegression with multiple methods...
Analyzing estimator: LinearSVC
Analyzing LinearSVC with coefficient and permutation importance...
Analyzing coefficients for Voting Ensemble_LinearSVC...
Coefficient importance saved to saved_final_models/final_models_20260615_104839/shap_analysis_results/Voting_Ensemble/LinearSVC
Analyzing estimator: SGDClassifier
Analyzing SGDClassifier with coefficient importance...
Analyzing coefficients for Voting Ensemble_SGDClassifier...
Coefficient importance saved to saved_final_models/final_models_20260615_104839/shap_analysis_results/Voting_Ensemble/SGDClassifier
Analyzed 3/3 estimators successfully
